# Etapa 2 - Enriquecimento sintetico

**Datathon 7MLET - Grupo 87** | Camada de experimentacao adaptativa

Este notebook cria a camada sintetica pedida na Etapa 2 e grava os artefatos em `data/synthetic_enrichment/`, mantendo separacao fisica em relacao a `data/kaggle/raw/`. A base de partida é o dataset processado da Etapa 1.

O que é gerado aqui:

1. `offer_catalog.parquet` - catalogo sintetico de bracos/ofertas.
2. `offer_events.parquet` - eventos de impressao com contexto de decisao e recompensa intermediaria.
3. `delayed_rewards.parquet` - log da recompensa final atrasada com censura pelo horizonte temporal.

A documentacao detalhada do schema fica em [`data/synthetic_enrichment/README.md`](../data/synthetic_enrichment/README.md).

## Escopo e criterio de geracao

A simulacao usa sementes controladas, horizonte de `14` dias e um proxy de conversao baseado em `subscribed`. A politica é estocástica, mas totalmente reprodutivel.

A ideia nao é reproduzir o mundo real de forma causal perfeita; é criar uma camada de experimentação consistente para a Etapa 3, com bracos, contexto, recompensa intermediaria e recompensa atrasada claramente separados.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.build_processed import load_processed

SYNTH_DIR = PROJECT_ROOT / 'data' / 'synthetic_enrichment'
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = {
    'catalog': 20240622,
    'policy': 20240623,
    'rewards': 20240624,
}
HORIZON_DAYS = 14
START_TS = pd.Timestamp('2013-01-07 08:00:00')
POLICY_TEMPERATURE = 0.75
DECISIONS_PER_DAY = 450

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print(f'Projeto: {PROJECT_ROOT}')
print(f'Saida sintetica: {SYNTH_DIR}')
print(f'Sementes: {SEEDS}')

Projeto: /home/higor/projects/fiap/datathon-7mlet-grupo-87
Saida sintetica: /home/higor/projects/fiap/datathon-7mlet-grupo-87/data/synthetic_enrichment
Sementes: {'catalog': 20240622, 'policy': 20240623, 'rewards': 20240624}


In [ ]:
df = load_processed().reset_index(drop=True).copy()
df['customer_id'] = np.arange(len(df), dtype='int64')

df['segment_age_band'] = pd.cut(
    df['age'],
    bins=[16, 25, 35, 45, 55, 65, 100],
    labels=['17-25', '26-35', '36-45', '46-55', '56-65', '66+'],
    include_lowest=True,
).astype(str)

df['segment_history'] = np.where(df['pdays'].to_numpy() == 999, 'cold_start', 'warm_history')
df['segment_macro_regime'] = np.select(
    [
        df['emp_var_rate'].to_numpy() <= -2.0,
        df['euribor3m'].to_numpy() >= 4.5,
    ],
    ['stress', 'tight'],
    default='neutral',
)
df['decision_ts'] = START_TS + pd.to_timedelta(df['customer_id'] // DECISIONS_PER_DAY, unit='D') + pd.to_timedelta(df['customer_id'] % 8, unit='h')

catalog = pd.DataFrame([
    {
        'arm_id': 'arm_control',
        'arm_name': 'Mensagem neutra',
        'channel': 'email',
        'objective': 'benchmark',
        'value_proposition': 'sem incentivo extra',
        'target_segment': 'geral',
        'base_score': 0.10,
        'delay_mean_days': 5.0,
        'delay_std_days': 2.0,
        'horizon_days': HORIZON_DAYS,
        'notes': 'braco de referencia para comparacao',
    },
    {
        'arm_id': 'arm_rate_boost',
        'arm_name': 'Taxa bonificada',
        'channel': 'call',
        'objective': 'conversion',
        'value_proposition': 'estimulo financeiro direto',
        'target_segment': 'contatos com maior resposta',
        'base_score': 0.28,
        'delay_mean_days': 7.0,
        'delay_std_days': 3.0,
        'horizon_days': HORIZON_DAYS,
        'notes': 'mais forte em celular e sinais de sucesso previo',
    },
    {
        'arm_id': 'arm_digital_bundle',
        'arm_name': 'Pacote digital',
        'channel': 'sms',
        'objective': 'activation',
        'value_proposition': 'adesao rapida e baixo atrito',
        'target_segment': 'publico mais jovem',
        'base_score': 0.24,
        'delay_mean_days': 4.0,
        'delay_std_days': 2.0,
        'horizon_days': HORIZON_DAYS,
        'notes': 'prioriza perfis jovens e contato celular',
    },
    {
        'arm_id': 'arm_consultative_call',
        'arm_name': 'Contato consultivo',
        'channel': 'call',
        'objective': 'conversion',
        'value_proposition': 'atendimento humano guiado por contexto',
        'target_segment': 'historico quente',
        'base_score': 0.26,
        'delay_mean_days': 8.0,
        'delay_std_days': 3.0,
        'horizon_days': HORIZON_DAYS,
        'notes': 'favorece contatos repetidos e sucesso anterior',
    },
    {
        'arm_id': 'arm_retention_plus',
        'arm_name': 'Oferta de relacionamento',
        'channel': 'push',
        'objective': 'retention',
        'value_proposition': 'beneficio progressivo e menor friccao',
        'target_segment': 'segmentos intermediarios',
        'base_score': 0.22,
        'delay_mean_days': 6.0,
        'delay_std_days': 2.0,
        'horizon_days': HORIZON_DAYS,
        'notes': 'equilibra conversao e relacionamento',
    },
])
catalog['generated_with_seed'] = SEEDS['catalog']
catalog = catalog[['arm_id', 'arm_name', 'channel', 'objective', 'value_proposition', 'target_segment', 'base_score', 'delay_mean_days', 'delay_std_days', 'horizon_days', 'notes', 'generated_with_seed']]
catalog.to_parquet(SYNTH_DIR / 'offer_catalog.parquet', index=False)

age = df['age'].to_numpy()
young = (age < 35).astype(float)
mid = ((age >= 35) & (age < 55)).astype(float)
senior = (age >= 55).astype(float)
cellular = df['contact'].eq('cellular').astype(float).to_numpy()
telephone = df['contact'].eq('telephone').astype(float).to_numpy()
success = df['poutcome'].eq('success').astype(float).to_numpy()
repeat_contact = (df['pdays'].to_numpy() != 999).astype(float)
previous_contact = (df['previous'].to_numpy() > 0).astype(float)
student = df['job'].eq('student').astype(float).to_numpy()
retired = df['job'].eq('retired').astype(float).to_numpy()
month_peak = df['month'].isin(['mar', 'apr', 'sep', 'oct', 'dec']).astype(float).to_numpy()
macro_stress = (df['segment_macro_regime'].eq('stress')).astype(float).to_numpy()
macro_tight = (df['segment_macro_regime'].eq('tight')).astype(float).to_numpy()

scores = np.column_stack([
    0.10 + 0.10 * repeat_contact + 0.05 * macro_stress,
    0.28 + 0.35 * cellular + 0.20 * success + 0.12 * macro_tight + 0.08 * young,
    0.24 + 0.25 * young + 0.20 * cellular + 0.10 * month_peak + 0.10 * student,
    0.26 + 0.25 * telephone + 0.15 * repeat_contact + 0.18 * success + 0.12 * retired,
    0.22 + 0.20 * mid + 0.12 * previous_contact + 0.08 * macro_stress + 0.08 * month_peak,
 ])

shifted = scores / POLICY_TEMPERATURE
shifted = shifted - shifted.max(axis=1, keepdims=True)
weights = np.exp(shifted)
prob_matrix = weights / weights.sum(axis=1, keepdims=True)

policy_rng = np.random.default_rng(SEEDS['policy'])
chosen_idx = np.empty(len(df), dtype=np.int64)
chosen_probability = np.empty(len(df), dtype=np.float64)
exploration_flag = np.empty(len(df), dtype=bool)

for i, row_prob in enumerate(prob_matrix):
    arm_idx = policy_rng.choice(len(catalog), p=row_prob)
    chosen_idx[i] = arm_idx
    chosen_probability[i] = row_prob[arm_idx]
    exploration_flag[i] = arm_idx != row_prob.argmax()

arm_lookup = catalog.iloc[chosen_idx].reset_index(drop=True)

events = pd.DataFrame({
    'impression_id': np.arange(1, len(df) + 1, dtype='int64'),
    'customer_id': df['customer_id'].to_numpy(),
    'decision_ts': df['decision_ts'].to_numpy(),
    'arm_id': arm_lookup['arm_id'].to_numpy(),
    'arm_name': arm_lookup['arm_name'].to_numpy(),
    'channel': arm_lookup['channel'].to_numpy(),
    'objective': arm_lookup['objective'].to_numpy(),
    'selection_probability': chosen_probability.round(6),
    'exploration_flag': exploration_flag,
    'context_score': scores[np.arange(len(df)), chosen_idx].round(6),
    'segment_age_band': df['segment_age_band'].to_numpy(),
    'segment_history': df['segment_history'].to_numpy(),
    'segment_macro_regime': df['segment_macro_regime'].to_numpy(),
    'poutcome': df['poutcome'].astype(str).to_numpy(),
    'contact': df['contact'].astype(str).to_numpy(),
    'job': df['job'].astype(str).to_numpy(),
    'subscribed': df['subscribed'].astype('int8').to_numpy(),
})

chosen_arm_signal = np.array([0.00, 0.10, 0.08, 0.12, 0.07])[chosen_idx]
intermediate_score = (
    -2.00
    + 1.00 * events['selection_probability'].to_numpy()
    + 0.45 * cellular
    + 0.35 * success
    + 0.20 * repeat_contact
    + 0.12 * young
    + chosen_arm_signal
)
intermediate_probability = 1 / (1 + np.exp(-intermediate_score))
reward_rng = np.random.default_rng(SEEDS['rewards'])
intermediate_reward = (reward_rng.random(len(df)) < intermediate_probability).astype('int8')
intermediate_delay = np.full(len(df), np.nan)
if intermediate_reward.sum() > 0:
    intermediate_delay[intermediate_reward == 1] = reward_rng.integers(0, 3, size=int(intermediate_reward.sum()))

events['reward_horizon_days'] = HORIZON_DAYS
events['intermediate_reward_probability'] = intermediate_probability.round(6)
events['intermediate_reward'] = intermediate_reward
events['intermediate_reward_delay_days'] = intermediate_delay

positive = df['subscribed'].to_numpy(dtype='int8') == 1
delay_draw = reward_rng.normal(
    loc=arm_lookup['delay_mean_days'].to_numpy(),
    scale=arm_lookup['delay_std_days'].to_numpy(),
)
delay_adjust = np.where(macro_tight == 1, -1.0, 0.0) + np.where(success == 1, -1.0, 0.0) + np.where(repeat_contact == 1, 0.5, 0.0)
reward_delay_days = np.clip(np.rint(delay_draw + delay_adjust), 0, HORIZON_DAYS + 7).astype('int16')

delayed_rewards = events[['impression_id', 'customer_id', 'decision_ts', 'arm_id', 'arm_name']].copy()
delayed_rewards['reward_value'] = 0
delayed_rewards['reward_delay_days'] = np.nan
delayed_rewards['reward_observed_at'] = pd.NaT
delayed_rewards['within_horizon'] = False
delayed_rewards['reward_status'] = 'no_conversion'

delayed_rewards.loc[positive, 'reward_value'] = 1
delayed_rewards.loc[positive, 'reward_delay_days'] = reward_delay_days[positive]
delayed_rewards.loc[positive, 'reward_observed_at'] = delayed_rewards.loc[positive, 'decision_ts'] + pd.to_timedelta(reward_delay_days[positive], unit='D')
delayed_rewards.loc[positive, 'within_horizon'] = reward_delay_days[positive] <= HORIZON_DAYS

observed_mask = positive & delayed_rewards['within_horizon'].to_numpy()
censored_mask = positive & ~delayed_rewards['within_horizon'].to_numpy()
delayed_rewards.loc[observed_mask, 'reward_status'] = 'observed'
delayed_rewards.loc[censored_mask, 'reward_status'] = 'censored'
delayed_rewards['horizon_days'] = HORIZON_DAYS

events.to_parquet(SYNTH_DIR / 'offer_events.parquet', index=False)
delayed_rewards.to_parquet(SYNTH_DIR / 'delayed_rewards.parquet', index=False)

print('Arquivos gerados:')
for file_path in sorted(SYNTH_DIR.glob('*.parquet')):
    print(f' - {file_path.name}')

print('\nCatalogo sintetico:')
print(catalog.to_string(index=False))

print('\nResumo dos eventos:')
print(f" - impresses: {len(events):,}".replace(',', '.'))
selected_arm = events['arm_name'].value_counts().idxmax()
exploration_rate = events['exploration_flag'].mean() * 100
intermediate_positive_rate = events['intermediate_reward'].mean() * 100
print(f' - arm mais escolhido: {selected_arm}')
print(f' - taxa de exploracao: {exploration_rate:.2f}%')
print(f' - recompensa intermediaria positiva: {intermediate_positive_rate:.2f}%')

print('\nResumo das recompensas atrasadas:')
status_counts = delayed_rewards['reward_status'].value_counts()
for status, count in status_counts.items():
    print(f' - {status}: {count}')

positive_rows = delayed_rewards['reward_value'].eq(1)
if positive_rows.any():
    positive_delay_mean = delayed_rewards.loc[positive_rows, 'reward_delay_days'].mean()
    within_horizon_rate = delayed_rewards.loc[positive_rows, 'within_horizon'].mean() * 100
    print(f' - atraso medio (positivos): {positive_delay_mean:.2f} dias')
    print(f' - dentro do horizonte: {within_horizon_rate:.2f}%')

print('\nSeparacao fisica confirmada: a pasta sintetica não referencia o CSV bruto do Kaggle e fica em data/synthetic_enrichment/.')

Arquivos gerados:
 - delayed_rewards.parquet
 - offer_catalog.parquet
 - offer_events.parquet

Catalogo sintetico:
               arm_id                 arm_name channel  objective                      value_proposition              target_segment  base_score  delay_mean_days  delay_std_days  horizon_days                                            notes  generated_with_seed
          arm_control          Mensagem neutra   email  benchmark                    sem incentivo extra                       geral        0.10              5.0             2.0            14              braco de referencia para comparacao             20240622
       arm_rate_boost          Taxa bonificada    call conversion             estimulo financeiro direto contatos com maior resposta        0.28              7.0             3.0            14 mais forte em celular e sinais de sucesso previo             20240622
   arm_digital_bundle           Pacote digital     sms activation           adesao rapida e baixo a

## Schema e horizonte temporal

Os detalhes completos do schema ficam no README da pasta sintetica. O ponto principal da Etapa 2 e que o catalogo de bracos, os eventos de impressao e as recompensas atrasadas sao artefatos distintos, gerados a partir da base processada, e nao do CSV bruto.

- `offer_catalog`: define os bracos/ofertas, o canal e o horizonte maximo.
- `offer_events`: registra a decisao, o contexto e a recompensa intermediaria.
- `delayed_rewards`: registra a recompensa final, o atraso e a censura.

A janela de observacao da recompensa final e de `14` dias. Eventos positivos fora dessa janela sao tratados como `censored`.

In [4]:
catalog_check = pd.read_parquet(SYNTH_DIR / 'offer_catalog.parquet')
events_check = pd.read_parquet(SYNTH_DIR / 'offer_events.parquet')
delayed_check = pd.read_parquet(SYNTH_DIR / 'delayed_rewards.parquet')

print('Validacao rapida:')
print(f' - catalogo: {catalog_check.shape[0]} linhas x {catalog_check.shape[1]} colunas')
print(f' - eventos: {events_check.shape[0]} linhas x {events_check.shape[1]} colunas')
print(f' - recompensas atrasadas: {delayed_check.shape[0]} linhas x {delayed_check.shape[1]} colunas')

print('\nPrimeiras linhas do catalogo:')
print(catalog_check.head().to_string(index=False))

print('\nDistribuicao dos bracos escolhidos:')
print(events_check['arm_name'].value_counts().to_string())

print('\nDistribuicao do status da recompensa:')
print(delayed_check['reward_status'].value_counts().to_string())

Validacao rapida:
 - catalogo: 5 linhas x 12 colunas
 - eventos: 39404 linhas x 21 colunas
 - recompensas atrasadas: 39404 linhas x 11 colunas

Primeiras linhas do catalogo:
               arm_id                 arm_name channel  objective                      value_proposition              target_segment  base_score  delay_mean_days  delay_std_days  horizon_days                                            notes  generated_with_seed
          arm_control          Mensagem neutra   email  benchmark                    sem incentivo extra                       geral        0.10              5.0             2.0            14              braco de referencia para comparacao             20240622
       arm_rate_boost          Taxa bonificada    call conversion             estimulo financeiro direto contatos com maior resposta        0.28              7.0             3.0            14 mais forte em celular e sinais de sucesso previo             20240622
   arm_digital_bundle           Pacote d

## Como reproduzir

1. Rodar a Etapa 1 para garantir `data/processed/bank_marketing.parquet`.
2. Executar este notebook.
3. Confirmar a saida em `data/synthetic_enrichment/`.

Comando recomendado para execucao automatica:

```bash
poetry run jupyter nbconvert --to notebook --execute --inplace notebooks/02_enriquecimento_sintetico.ipynb
```